# Assignment 2: Gradio dashboard

You're using a notebook for this stage. If you'd rather use a script,
write `app.py` instead and delete this file. `main.py` refuses to run if
it finds both `app.py` and `app.ipynb`, so exactly one of them must exist.

Replace this cell and everything below it with your own code. When run
via `uv run python main.py app`, this notebook must build and launch a Gradio app
(a `demo` that calls `.launch()`) using only what your training stage already
produced. It must never train anything itself. Load the committed weights in
`weights/`, never whatever the training run left in `runs/`.

At minimum, the dashboard must let you:

- Upload an image and see the model's predictions at the current threshold.
- Move the two edges of your escalation band and watch the confusion matrix,
  the expected cost per 1,000 units, and the share of cases escalated to a
  human all change together.
- See the cost-against-threshold curve with your chosen operating point marked
  on it.
- See per-class performance, not just the overall number.

How you structure the code beyond that is your call to make and be able to
defend: file layout, function boundaries, naming.

In [ ]:
import json
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import gradio as gr
import numpy as np
import pandas as pd
from ultralytics import YOLO

import costing

ARTIFACTS = Path("artifacts")
results = json.loads((ARTIFACTS / "results.json").read_text())
scores = pd.read_csv(ARTIFACTS / "test_scores.csv")
sweep = pd.read_csv(ARTIFACTS / "threshold_sweep.csv")
model = YOLO("weights/best.pt")

COSTS = results["costs"]
BASE_RATE = results["base_rate"]
OP = results["operating_point"]
POSITIVE = results["dataset"]["positive_classes"]
y = scores["is_fall"].astype(bool).to_numpy()
s = scores["score_finetuned"].to_numpy()

DEFAULT_IMAGES_PER_NIGHT = 28800
DEFAULT_SECONDS_PER_REVIEW = 15

In [ ]:
per_class_table = pd.DataFrame(results["detection"]["per_class"]).round(3)

baseline_table = pd.DataFrame(results["baselines"]).T.reset_index(names="model")
baseline_table = baseline_table[["model", "tp", "fp", "fn", "tn", "recall", "precision", "average_precision"]]
baseline_table["model"] = baseline_table["model"].map(
    {"finetuned": "Fine-tuned yolo11n", "coco": "COCO zero-shot (person wider than tall)",
     "trivial": "Trivial (predict nothing)"}
)
baseline_table[["tp", "fp", "fn", "tn"]] = baseline_table[["tp", "fp", "fn", "tn"]].astype(int)
baseline_table = baseline_table.round(3)

images_per_fall = results.get("images_per_fall", 1)
incident = results.get("incident_costs", {})
cost_table = pd.DataFrame([
    {"outcome": "Missed fall (false negative)", "per incident (EUR)": incident.get("fn"),
     "per image (EUR)": COSTS["fn"]},
    {"outcome": "Caught fall (true positive)", "per incident (EUR)": incident.get("tp"),
     "per image (EUR)": COSTS["tp"]},
    {"outcome": "False alert (false positive)", "per incident (EUR)": None, "per image (EUR)": COSTS["fp"]},
    {"outcome": "Correctly ignored (true negative)", "per incident (EUR)": None, "per image (EUR)": COSTS["tn"]},
    {"outcome": "One operator review", "per incident (EUR)": None, "per image (EUR)": COSTS["review"]},
]).round(4)

assumptions_md = (
    f"**Base rate:** {BASE_RATE:.7f} of images show a fall "
    f"(each fall is visible in about {images_per_fall} images). "
    f"**Chosen operating point:** single threshold {OP['single_threshold']:.2f}; "
    f"cheapest review band {OP['review_low']:.2f} to {OP['review_high']:.2f}."
)
sensitivity_table = pd.DataFrame(results["sensitivity"])
sensitivity_table["base_rate"] = sensitivity_table["base_rate"].map(lambda p: f"{p:.4%}")
sensitivity_table = sensitivity_table.round(4)

In [ ]:
def band_outcomes(low, high):
    passed, reviewed, alerted = costing.route(s, low, high)
    return pd.DataFrame(
        {
            "Pass": [int((passed & y).sum()), int((passed & ~y).sum())],
            "Review": [int((reviewed & y).sum()), int((reviewed & ~y).sum())],
            "Alert": [int((alerted & y).sum()), int((alerted & ~y).sum())],
        },
        index=["True fall", "No fall"],
    ).reset_index(names="test images")


def cost_curve(low, high):
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.plot(sweep["threshold"], sweep["cost_per_1000"], label="single threshold")
    ax.axvline(OP["single_threshold"], color="red", linestyle="--",
               label=f"chosen = {OP['single_threshold']:.2f}")
    ax.axvspan(low, high, color="orange", alpha=0.25, label="current review band")
    ax.set_yscale("log")
    ax.set_xlabel("Confidence threshold")
    ax.set_ylabel("Expected cost per 1,000 images (EUR, log scale)")
    ax.set_title("Cost against threshold (fine-tuned model)")
    ax.legend()
    fig.tight_layout()
    plt.close(fig)
    return fig


def update_band(low, high, images_per_night, seconds_per_review):
    low, high = sorted((float(low), float(high)))
    cost, review_share = costing.cost_per_1000(s, y, low, high, COSTS, BASE_RATE)
    per_night = images_per_night / 1000
    reviews_per_night = review_share * images_per_night
    operator_hours = reviews_per_night * seconds_per_review / 3600

    alternatives = pd.DataFrame([
        {"design": "Do nothing", "EUR per 1,000 images": results["alternatives"]["do_nothing"]},
        {"design": "Person checks every image", "EUR per 1,000 images": results["alternatives"]["human_checks_everything"]},
        {"design": f"Model alone at {OP['single_threshold']:.2f}", "EUR per 1,000 images": results["alternatives"]["model_alone"]},
        {"design": f"Current band {low:.2f} to {high:.2f}", "EUR per 1,000 images": cost},
    ])
    alternatives["EUR per gym per night"] = alternatives["EUR per 1,000 images"] * per_night
    alternatives = alternatives.round(2)

    summary = (
        f"### Band {low:.2f} to {high:.2f}\n"
        f"- **Expected cost:** EUR {cost:.2f} per 1,000 images "
        f"(EUR {cost * per_night:.2f} per gym per night)\n"
        f"- **Share sent for review:** {review_share:.2%} of images\n"
        f"- **Review load:** {reviews_per_night:.0f} checks per gym per night, "
        f"about {operator_hours:.1f} operator hours"
    )
    return summary, band_outcomes(low, high), alternatives, cost_curve(low, high)


def analyse_image(image, low, high):
    if image is None:
        return None, "Upload an image to see the model's predictions."
    low, high = sorted((float(low), float(high)))
    r = model.predict(image, conf=max(low, 0.01), verbose=False)[0]
    fall_scores = [float(c) for c, k in zip(r.boxes.conf, r.boxes.cls) if r.names[int(k)] in POSITIVE]
    score = max(fall_scores, default=0.0)
    if score >= high:
        decision = "ALERT: high-priority alert to the operator"
    elif score >= low:
        decision = "REVIEW: the operator checks this camera"
    else:
        decision = "PASS: nothing happens"
    text = f"**Fall score:** {score:.2f}  \n**Decision:** {decision}"
    return r.plot()[:, :, ::-1], text

In [ ]:
with gr.Blocks(title="Gym fall detection") as demo:
    gr.Markdown("# Fall detection for unstaffed 24/7 gyms")
    gr.Markdown(assumptions_md)
    with gr.Row():
        low_slider = gr.Slider(0, 1, value=OP["review_low"], step=0.01, label="Low threshold (below: pass)")
        high_slider = gr.Slider(0, 1, value=OP["review_high"], step=0.01, label="High threshold (at or above: alert)")

    with gr.Tab("Decision band"):
        with gr.Row():
            images_input = gr.Number(value=DEFAULT_IMAGES_PER_NIGHT, label="Images per gym per night")
            seconds_input = gr.Number(value=DEFAULT_SECONDS_PER_REVIEW, label="Seconds per operator review")
        summary_out = gr.Markdown()
        with gr.Row():
            outcomes_out = gr.Dataframe(label="Test images by outcome (confusion matrix for the band)")
            alternatives_out = gr.Dataframe(label="Priced against the alternatives")
        curve_out = gr.Plot(label="Cost against threshold")

    with gr.Tab("Try an image"):
        with gr.Row():
            image_in = gr.Image(type="pil", label="Upload a camera image")
            image_out = gr.Image(label="Predictions at the current thresholds")
        decision_out = gr.Markdown()

    with gr.Tab("Model performance"):
        gr.Markdown("### Per-class results on the test split")
        gr.Dataframe(value=per_class_table, interactive=False)
        gr.Markdown("### Against the baselines (image level, threshold 0.5)")
        gr.Dataframe(value=baseline_table, interactive=False)
        gr.Markdown(f"### Detection confusion matrix (boxes with confidence of at least "
                    f"{results['detection'].get('confusion_conf', 0.25)})")
        gr.Image(value=str(ARTIFACTS / "confusion_matrix.png"), interactive=False, show_label=False)

    with gr.Tab("Costs and assumptions"):
        gr.Dataframe(value=cost_table, interactive=False)
        gr.Markdown(assumptions_md)
        gr.Markdown("### How the best threshold moves with the base rate")
        gr.Dataframe(value=sensitivity_table, interactive=False)

    band_inputs = [low_slider, high_slider, images_input, seconds_input]
    band_outputs = [summary_out, outcomes_out, alternatives_out, curve_out]
    for component in band_inputs:
        component.change(update_band, band_inputs, band_outputs)
    for component in [image_in, low_slider, high_slider]:
        component.change(analyse_image, [image_in, low_slider, high_slider], [image_out, decision_out])
    demo.load(update_band, band_inputs, band_outputs)

demo.launch()